In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

## Overall test / assertion / generalization exclusions

In [ ]:
import pandas as pd
import re

query = f"SELECT * FROM mv_exclusions_all"
df = pd.read_sql_query(query, conn)

# Adapt the variant names.
def format_variant(variant):
    return re.sub(r'_(\d+)_TRIES$', r'$_{\1}$', variant)

df['variant'] = df['variant'].apply(format_variant)

# Map 'level' to 'Type'.
level_map = {
    '1-TEST': 'Test',
    '2-ASSERTION': 'Assertion',
    '3-GENERALIZATION': 'Generalization'
}
df['Type'] = df['level'].map(level_map)

df = df[['variant', 'Type', 'is_included', 'excluded_by', 'count']]

display(df)

# Get unique (variant, level) pairs in original order.
ordered_pairs = df[['variant', 'Type']].drop_duplicates()

# Calculate included and excluded counts.
included = df[df['is_included'] == True].groupby(['variant', 'Type'])['count'].sum().reset_index()
excluded = df[df['is_included'] == False].groupby(['variant', 'Type'])['count'].sum().reset_index()

included = included.rename(columns={'count': 'included_count'})
excluded = excluded.rename(columns={'count': 'excluded_count'})

# Merge counts.
result = pd.merge(ordered_pairs, included, on=['variant', 'Type'], how='left')
result = pd.merge(result, excluded, on=['variant', 'Type'], how='left')

# Fill NaNs with 0 and ensure integer type.
result['included_count'] = result['included_count'].fillna(0).astype(int)
result['excluded_count'] = result['excluded_count'].fillna(0).astype(int)

# Compute Total column
result['Total'] = result['included_count'] + result['excluded_count']

result['included_pct'] = (result['included_count'] / result['Total'] * 100).round(1)
result['excluded_pct'] = (result['excluded_count'] / result['Total'] * 100).round(1)

# Format percentages
def format_pct(val):
    return f"{val:04.1f}"

result['included_pct_str'] = result['included_pct'].apply(format_pct)
result['excluded_pct_str'] = result['excluded_pct'].apply(format_pct)

# Format Included and Excluded columns as "count (pct%)"
result['Included'] = result.apply(lambda row: f"{row['included_count']}\\; ({row['included_pct_str']}\\%)", axis=1)
result['Excluded'] = result.apply(lambda row: f"{row['excluded_count']}\\; ({row['excluded_pct_str']}\\%)", axis=1)

display(result[['variant', 'Type', 'Total', 'Included', 'Excluded']])

# Build LaTeX table
lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"  \caption{Included and excluded counts by variant and level.}")
lines.append(r"  \label{tab:exclusions-summary}")
lines.append(r"  \begin{tabular}{llrrr}")  # Hardcoded alignments
lines.append(r"    \toprule")
lines.append(r"    Variant & Type & Total & \multicolumn{1}{c}{Included} & \multicolumn{1}{c}{Excluded} \\")  # Hardcoded headers
lines.append(r"    \midrule")

for _, row in result.iterrows():
    row_str = ' & '.join(str(row[col]) for col in ['variant', 'Type', 'Total', 'Included', 'Excluded']) + r' \\'
    lines.append(f"    {row_str}")

lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

## Filtering-based test / assertion / generalization exclusions

In [ ]:
import pandas as pd
import re

# Query and DataFrame creation
query = "SELECT * FROM mv_exclusions_filtering WHERE reject > 0"
df = pd.read_sql_query(query, conn)

# Ensure integer columns are int type
df['total'] = df['total'].astype(int)
df['accept'] = df['accept'].astype(int)
df['reject'] = df['reject'].astype(int)

# Map 'level' to 'Type'
level_map = {
    '1-TEST': 'Test',
    '2-ASSERTION': 'Assertion',
    '3-GENERALIZATION': 'Generalization'
}
df['Type'] = df['level'].map(level_map)

# Format 'variant' for LaTeX subscripts
def format_variant(variant):
    return re.sub(r'_(\d+)_TRIES$', r'$_{\1}$', variant)

df['Variant'] = df['variant'].apply(format_variant)

# Add 'Defer' to 'Reject' and drop 'Defer'
df['reject'] = df['reject'] + df['defer']
df = df.drop(columns=['defer'])

# Remove 'Filter' suffix from filter names
df['filter_name'] = df['filter_name'].str.replace(r'Filter$', '', regex=True)

# Calculate percentages
df['accept_pct'] = (df['accept'] / df['total'] * 100).round(1)
df['reject_pct'] = (df['reject'] / df['total'] * 100).round(1)

def format_count_pct(count, pct):
    return f"{count}\\; ({pct:04.1f}\\%)"

df['Accept'] = df.apply(lambda row: format_count_pct(row['accept'], row['accept_pct']), axis=1)
df['Reject'] = df.apply(lambda row: format_count_pct(row['reject'], row['reject_pct']), axis=1)

display(df[['Variant', 'Type', 'filter_name', 'total', 'Accept', 'Reject']])

# Build LaTeX table
lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"  \caption{Filtering results for tests, assertions, and generalizations by filter and (generalization) variant.}")
lines.append(r"  \label{tab:exclusions-filtering}")
lines.append(r"  \begin{tabular}{lllrrr}")  # Hardcoded alignments
lines.append(r"    \toprule")
lines.append(r"    Variant & Type & Filter Name & Total & \multicolumn{1}{c}{Accept} & \multicolumn{1}{c}{Reject} \\")  # Hardcoded headers
lines.append(r"    \midrule")

prev_type = df.iloc[0]['Type']
for i, row in df.iterrows():
    # Insert \midrule when Type changes (but not before the first group)
    if i > 0 and row['Type'] != prev_type:
        lines.append(r"    \midrule")
    prev_type = row['Type']
    # Hardcoded column order
    row_str = ' & '.join(str(row[col]) for col in ['Variant', 'Type', 'filter_name', 'total', 'Accept', 'Reject']) + r' \\'
    lines.append(f"    {row_str}")

lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)

## Exclusions caused by SPF execution failures

In [ ]:
import pandas as pd

query = f"SELECT * FROM mv_exclusions_jpf"
df = pd.read_sql_query(query, conn)

category_map = {
    "ArithmeticException: div by 0": "SPF exception",
    "NoSuchMethodException": "SPF exception",
    "AssertionFailedError": "SPF exception",
    "RuntimeException: symbolic array length": "SPF exception",
    "NoUncaughtExceptionsProperty": "SPF exception",
    "ArrayIndexOutOfBoundsException (setDoubleValue)": "SPF exception",
    "ArrayIndexOutOfBoundsException (simple)": "SPF exception",
    "NullPointerException (queueMark)": "SPF exception",
    "ArrayIndexOutOfBoundsException (setLongValue)": "SPF exception",
    "NullPointerException (writeSpecificationFiles:147)": "Teralizer exception",
    "NullPointerException (writeSpecificationFiles:124)": "Teralizer exception",
    "Failed to collect specification": "Teralizer exception",
    "OutOfMemoryError: Java heap space": "OutOfMemoryError",
    "OutOfMemoryError: GC overhead": "OutOfMemoryError",
    # The following are left unchanged:
    # "PC size limit exceeded"
    # "Depth limit exceeded"
    # "Execution timeout"
}

# Apply the mapping, keeping unmapped categories as is
df['merged_category'] = df['error_category'].map(category_map).fillna(df['error_category'])

display(df[['error_category', 'merged_category', 'count']])

# Group by merged_category and sum the counts
df_merged = df.groupby('merged_category', as_index=False)['count'].sum()

# Sort by 'count' in descending order
df_merged = df_merged.sort_values(by='count', ascending=False).reset_index(drop=True)

# Add percent column
total = df_merged['count'].sum()
df_merged['percent'] = (df_merged['count'] / total * 100).round(2)

# Rename columns
df_merged.rename(columns={'merged_category': 'Error Type', 'count': 'Total', 'percent': 'Percent'}, inplace=True)

display(df_merged)

# Generate LaTeX code:
latex_table = [
    r"\begin{table}[H]",
    r"  \centering",
    r"  \caption{Number of SPF execution failures by error type.}",
    r"  \label{tab:exclusions-spf}",
    f"  \\begin{{tabular}}{{{'l' + 'r' * (len(df_merged.columns) - 1)}}}",
    r"    \toprule",
    "    " + " & ".join(df_merged.columns) + r" \\",
    r"    \midrule"
]

for _, row in df_merged.iterrows():
    values = [
        str(row[col]) if df_merged[col].dtype == "object"
        else f"{row[col]:.2f}" if isinstance(row[col], float)
        else str(row[col])
        for col in df_merged.columns
    ]
    latex_table.append("    " + " & ".join(values) + r" \\")

latex_table += [
    r"    \bottomrule",
    r"  \end{tabular}",
    r"\end{table}"
]

print("\n".join(latex_table))

## Exclusions caused by test failures

In [ ]:
import re

query = f"SELECT * FROM mv_exclusions_test_fails"
df = pd.read_sql_query(query, conn)

# Pivot as before
ordered_variants = (
    df[['variant', 'variant_order']]
    .drop_duplicates()
    .sort_values('variant_order')
    ['variant']
    .tolist()
)
pivoted_df = df.pivot(index='failure_type', columns='variant', values='count')
pivoted_df = pivoted_df[ordered_variants].fillna(0).astype(int)

display(pivoted_df)

# --- AUTOMATED HEADER GENERATION ---

# Extract base variant and tries
variant_info = []
for v in pivoted_df.columns:
    m = re.match(r'([A-Z]+)(?:_(\d+)_TRIES)?', v)
    if m:
        base = m.group(1)
        tries = m.group(2) if m.group(2) else '-'
        variant_info.append((v, base, tries))
    else:
        variant_info.append((v, v, '-'))

# Group columns by base variant
from collections import OrderedDict
grouped = OrderedDict()
for v, base, tries in variant_info:
    grouped.setdefault(base, []).append((v, tries))

# Build header rows
header1 = ['Variant']
header2 = ['Tries']
cmidrules = []
col_idx = 2  # LaTeX columns start at 1, first is 'Variant'

for base, cols in grouped.items():
    n = len(cols)
    if n == 1:
        header1.append(base)
        header2.append('-')
        # No cmidrule needed for single columns
        col_idx += 1
    else:
        header1 += [f'\\multicolumn{{{n}}}{{c}}{{{base}}}']
        header2 += [tries for _, tries in cols]
        # cmidrule for this group
        start = col_idx
        end = col_idx + n - 1
        cmidrules.append(f'\\cmidrule(lr){{{start}-{end}}}')
        col_idx += n

header1_line = ' & '.join(header1) + r' \\'
header2_line = ' & '.join(header2) + r' \\'
cmidrules_line = '\n    '.join(cmidrules)

# --- BUILD THE TABLE ---
latex_table = r"""\begin{table}[H]
  \caption{Number of test execution failures by exception type and (generalization) variant.}
  \label{tab:exclusions-test-fails}
  \begin{tabular}{l""" + "r" * (len(pivoted_df.columns)) + r"""}
    \toprule
    """ + header1_line + "\n    " + cmidrules_line + "\n    " + header2_line + r"""
    \midrule
"""

# Data rows
for failure_type, row in pivoted_df.iterrows():
    row_str = "    " + failure_type + " & " + " & ".join(str(x) for x in row.values) + r" \\"
    latex_table += row_str + "\n"

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)